# Data Cleaning and Text Preprocessing.

## Activity 1: Load and Review the Dataset

In [1]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print('All libraries imported')

All libraries imported


### Load the original dataset form daw/raw/

In [7]:
df = pd.read_csv('../data/raw/ted_talks.csv')

### Review the findings from Task 2
Review of Missing Values and Duplicates

The dataset contains missing values in four columns, with about_speaker having the highest proportion (19.26%). Missing values in about_talk, tags, and views are minimal. No exact duplicate rows were identified.

Missing descriptions and tags may affect content-based recommendations by reducing the information available for matching talks. Missing speaker descriptions may affect speaker-based recommendations. Missing view counts may influence popularity-based ranking.

Overall, the dataset is relatively complete, but missing speaker descriptions should be handled appropriately. Missing descriptions, tags, and view counts should also be considered during preprocessing. Further checks for empty strings and duplicate talks based on title or URL are recommended.

### Identify the fields that require cleaning.
The fields that require cleaning are:
1. about_talk – May contain missing or empty talk descriptions.
2. about_speaker – May have missing information about speakers.
3. tags – Requires standardizing tag formats and handling missing values.
4. duration – Needs validation and conversion into a consistent numeric format.
5. views – Requires removing commas and converting values to numeric data.
6. title – Should be checked for missing or empty talk titles.
7. transcript – Needs checking for availability and missing content.
8. URL – Should be checked for missing or invalid links.

### Preserve the original dataset without overwriting it

In [14]:
cleaned_df = df.copy()

## Activity 2: Handle Missing Values

### Identify missing or empty values in important columns

In [15]:
print(cleaned_df.columns.tolist())

['speaker_name', 'title', 'posted_date', 'duration', 'Link', 'about_speaker', 'about_talk', 'views', 'tags']


In [16]:
imp_cols = ['speaker_name', 'title', 'about_speaker', 'about_talk', 'tags']
for col in imp_cols:
    missing = cleaned_df[col].isna().sum()
    empty = cleaned_df[col].astype('string').str.strip().eq('').sum()

    print(f"{col}: Missing = {missing}, Empty = {empty}")

speaker_name: Missing = 0, Empty = 0
title: Missing = 0, Empty = 0
about_speaker: Missing = 416, Empty = 0
about_talk: Missing = 1, Empty = 0
tags: Missing = 7, Empty = 0


### Decide how to handle each issue based on the field's purpose

Missing values were assessed according to the purpose and importance of each field. The speaker_name and title columns contain no missing values, so no cleaning is required for these fields. The about_speaker column has 416 missing values, while about_talk has one and tags has seven. Since these columns provide supplementary descriptive information rather than essential identifiers, the missing values will be retained instead of deleting entire rows or filling them with potentially inaccurate information. This approach preserves the available data while maintaining its accuracy and integrity.

## Activity 3: Remove Duplicate Records

### Identify exact duplicate rows

In [17]:
exact_duplicates = cleaned_df.duplicated()
print("Exact duplicate rows identified:", exact_duplicates.sum())

Exact duplicate rows identified: 0


### Check for repeated talk identifiers or URLs, where available

In [21]:
print(cleaned_df[cleaned_df['Link'].duplicated(keep=False)])

Empty DataFrame
Columns: [speaker_name, title, posted_date, duration, Link, about_speaker, about_talk, views, tags]
Index: []


No duplicate records were found, hence no removal was done

## Activity 4: Clean Text Fields

### Converting text to lowercase where appropriate

In [26]:
text_cols = ['title', 'about_speaker', 'about_talk', 'speaker_name']
for col in text_cols:
    if col in cleaned_df.columns:
        cleaned_df[col] = cleaned_df[col].str.lower()
# lowercasing tags column
cleaned_df['tags'] = cleaned_df['tags'].apply(
    lambda tags: [tag.lower().strip() for tag in tags]if isinstance(tags, list) else tags)

### Removing unnecessary leading, trailing, and repeated whitespace.

In [29]:
for col in text_cols:
    cleaned_df[col] = cleaned_df[col].str.strip().str.replace(r'\s+', ' ', regex=True)
# for tags column
cleaned_df['tags'] = cleaned_df['tags'].apply(lambda tags: [tag.strip() for tag in tags] if isinstance(tags, list) else tags)

### Handling empty strings and missing text consistently.

In [30]:
for col in text_cols:
    cleaned_df[col] = cleaned_df[col].replace(r'^\s*$', pd.NA, regex=True)
# for tags column
cleaned_df['tags'] = cleaned_df['tags'].apply(lambda x: pd.NA if isinstance(x, list) and len(x) == 0 else x)

### Applying appropriate punctuation handling without damaging meaningful text

In [32]:
for col in ['title', 'about_speaker', 'about_talk', 'transcript']:
    if col in cleaned_df.columns:
        cleaned_df[col] = cleaned_df[col].str.replace(r'([!?.,])\1+', r'\1', regex=True)

## Activity 5: Create a Combined Text Field

In [33]:
text_cols = ['title', 'tags', 'about_talk', 'transcript']
text_cols = [col for col in text_cols if col in cleaned_df.columns]

cleaned_df['combined_text'] = cleaned_df[text_cols].fillna('').agg(' '.join, axis=1).str.strip()

## Activity 6: Save the Cleaned Dataset

In [34]:
text_cols = ['title', 'tags', 'about_talk', 'transcript']
text_cols = [col for col in text_cols if col in cleaned_df.columns]
cleaned_df['combined_text'] = cleaned_df[text_cols].fillna('').agg(' '.join, axis=1).str.strip()